In [97]:
import mariadb

conn = mariadb.connect(
    host="localhost",
    port=3306,
    user="root",
    password="",
    database="gym_2"
)

cursor = conn.cursor()

In [98]:
cursor.execute("SHOW TABLES")

tables = [row[0] for row in cursor.fetchall()]

print("Tables in gym:")
for table in tables:
    print(" -", table)

Tables in gym:
 - abonnements
 - activities
 - admins
 - assurances
 - categories
 - commandes
 - crenaus
 - decharges
 - entres
 - inscriptions
 - items
 - listes
 - membres
 - migrations
 - ouvertures
 - ouvertures2
 - password_resets
 - presences
 - produits
 - puces
 - settings
 - sorties
 - tarifs
 - tickets
 - users
 - versements


In [99]:
# cursor.execute("SELECT * FROM decharges")

# columns = [description[0] for description in cursor.description]

# print("Columns:")
# for column in columns:
#     print(" -", column)

# rows = cursor.fetchall()

# print(f"\nNumber of rows: {len(rows)}")

# for row in rows:
#     print(row)

assurances
commandes
membres


In [100]:
import pandas as pd
pd.set_option('display.max_columns', None)
query = "SELECT * FROM membres"

df_membres = pd.read_sql(query, conn)

C:\Users\zakmins\AppData\Local\Temp\ipykernel_15856\3281200361.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_membres = pd.read_sql(query, conn)


In [101]:
query = "SELECT * FROM inscriptions"

df_inscriptions = pd.read_sql(query, conn)

C:\Users\zakmins\AppData\Local\Temp\ipykernel_15856\1581818696.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_inscriptions = pd.read_sql(query, conn)


In [102]:
query = "SELECT * FROM presences"

df_presences = pd.read_sql(query, conn)

conn.close()

C:\Users\zakmins\AppData\Local\Temp\ipykernel_15856\4132726859.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_presences = pd.read_sql(query, conn)


In [103]:
df_presences.head()

,id,inscription,membre,created_at,updated_at,type,prix,activity,telephone,nom_prenom,user
0,1,1.0,1,2024-12-16 16:23:16,2024-12-16 16:23:16,1.0,NaN,None,None,None,NaN
1,2,2.0,2,2024-12-17 00:58:37,2024-12-17 00:58:37,1.0,NaN,None,None,None,NaN
2,3,2.0,2,2024-12-17 01:03:05,2024-12-17 01:03:05,NaN,NaN,None,None,None,NaN
3,12,4.0,4,2024-12-26 17:29:25,2024-12-26 17:29:25,1.0,NaN,None,None,None,NaN
4,5,2.0,2,2024-12-17 01:16:51,2024-12-17 01:16:51,1.0,NaN,None,None,None,NaN


In [104]:
df_presences = df_presences.loc[
    df_presences['membre'].isna() &
    df_presences['prix'].notna()
]

In [105]:
df_presences.head()

,id,inscription,membre,created_at,updated_at,type,prix,activity,telephone,nom_prenom,user
24,28,NaN,None,2025-01-04 15:43:22,2025-01-04 15:43:22,1.0,300.0,None,None,MAHREZ,53.0
27,31,NaN,None,2025-01-04 17:26:36,2025-01-04 17:26:36,1.0,300.0,None,None,JARI,53.0
28,32,NaN,None,2025-01-04 18:17:41,2025-01-04 18:17:41,1.0,300.0,None,None,BOUALI,53.0
31,35,NaN,None,2025-01-04 18:48:13,2025-01-04 18:48:13,1.0,300.0,None,None,BOUALI ADLEN,53.0
32,36,NaN,None,2025-01-04 18:54:45,2025-01-04 18:54:45,1.0,300.0,None,None,LEKHAL,53.0


In [106]:
PRESENCES_THRESHOLD_DATE = '2026-01-01'

df_presences = df_presences[df_presences['created_at'] >= PRESENCES_THRESHOLD_DATE]
df_presences['prix'] = df_presences['prix'].round().astype(int)

print(f"{len(df_presences)} guest session(s) since {PRESENCES_THRESHOLD_DATE}")
df_presences.head()

3358 guest session(s) since 2026-01-01


,id,inscription,membre,created_at,updated_at,type,prix,activity,telephone,nom_prenom,user
19824,19892,NaN,None,2026-01-01 09:15:49,2026-01-01 09:15:49,1.0,300,None,None,None,53.0
19832,19900,NaN,None,2026-01-01 09:56:26,2026-01-01 09:56:26,1.0,300,None,None,None,53.0
19840,19908,NaN,None,2026-01-01 11:10:09,2026-01-01 11:10:09,1.0,300,None,None,None,53.0
19843,19911,NaN,None,2026-01-01 12:12:02,2026-01-01 12:12:02,1.0,300,None,None,None,53.0
19852,19920,NaN,None,2026-01-01 14:47:35,2026-01-01 14:47:35,1.0,400,None,None,None,55.0


## Write df_presences into the live Smolympic SQLite database (guest sessions)

These are walk-in "free sessions" — one-off paid visits not tied to a member (`membre` is null, `prix` is set). Smolympic's `guest_sessions` table (see the *Guest sessions* panel in the live app) is much thinner than `presences`, so most columns have no home:

| df_presences column | Smolympic field |
|---|---|
| `nom_prenom` | `guest_sessions.name` (falls back to `"Invité"` if blank/null) |
| `prix` (rounded to int) | `guest_sessions.amount` **and** `payments.amount` |
| `created_at` | `guest_sessions.entry_time` **and** `payments.date` |
| — | `guest_sessions.expires_at` = `entry_time` + 2h (same rule the live app uses; these are historical so the window has long since lapsed) |

Mirroring exactly what the live "+ Session" walk-in flow does (`router.js` `addFreeSession`), each row also inserts a matching `payments` row (`kind='session'`, `sport='GYM'`, `method='Cash'`, `member_id=NULL`, `walk_in=1`) so this year's revenue and session-count reports include these historical guest sessions.

`id`, `inscription`, `membre`, `updated_at`, `type`, `activity`, `telephone`, `user` have no matching column and are **not** imported.

**No dedup / no re-run guard**: unlike the members import, `guest_sessions` has no natural unique key to check against, so this cell is **not** safe to re-run — running it twice will double-insert every row. Run it once; restore from the backup it prints if you need to redo it.

**Close the Smolympic app first**, same reason as the members import above.

In [107]:
import os, shutil, sqlite3
from datetime import datetime, timedelta

db_path = os.path.expandvars(r'%APPDATA%\SMOLYMPIC\smolympic.db')
assert os.path.exists(db_path), f"Smolympic database not found at {db_path}"

backup_path = f"{db_path}.bak-{datetime.now():%Y%m%d%H%M%S}"
shutil.copy2(db_path, backup_path)
print(f"Backed up live database to {backup_path}")


def clean_str(v):
    if v is None or pd.isna(v):
        return None
    v = str(v).strip()
    return v or None


def iso(dt):
    return dt.strftime('%Y-%m-%dT%H:%M:%S')


conn = sqlite3.connect(db_path)
conn.execute('PRAGMA foreign_keys = ON')
cur = conn.cursor()

inserted, fallback_named = 0, 0

try:
    for _, row in df_presences.iterrows():
        name = clean_str(row['nom_prenom'])
        if not name:
            name = 'Invité'
            fallback_named += 1

        amount = int(row['prix'])
        entry_iso = iso(row['created_at'])
        expires_iso = iso(row['created_at'] + timedelta(hours=2))

        cur.execute(
            'INSERT INTO guest_sessions (name,amount,entry_time,expires_at) VALUES (?,?,?,?)',
            (name, amount, entry_iso, expires_iso),
        )
        cur.execute(
            'INSERT INTO payments (member_id,amount,kind,sport,method,date,walk_in) VALUES (NULL,?,?,?,?,?,1)',
            (amount, 'session', 'GYM', 'Cash', entry_iso),
        )
        inserted += 1
except Exception:
    conn.rollback()
    conn.close()
    raise

conn.commit()
conn.close()

print(f"Inserted {inserted} guest session(s) (+ matching payment rows).")
if fallback_named:
    print(f"{fallback_named} row(s) had no nom_prenom and were named 'Invité'.")
print(f"A backup of the pre-import database was saved to: {backup_path}")

Backed up live database to C:\Users\zakmins\AppData\Roaming\SMOLYMPIC\smolympic.db.bak-20260823162751
Inserted 3358 guest session(s) (+ matching payment rows).
1900 row(s) had no nom_prenom and were named 'Invité'.
A backup of the pre-import database was saved to: C:\Users\zakmins\AppData\Roaming\SMOLYMPIC\smolympic.db.bak-20260823162751


In [108]:
df_membres.drop(columns=['photo', 'etat', 'email', 'identite', 'type', 'source', 'cn', 'dm', 'remarque', 'created_at', 'updated_at'], inplace=True)

In [109]:
df_membres.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1347 entries, 0 to 1346
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         1347 non-null   int64  
 1   nom        1347 non-null   object 
 2   prenom     1347 non-null   object 
 3   telephone  222 non-null    object 
 4   adresse    0 non-null      object 
 5   sexe       1347 non-null   object 
 6   naissance  609 non-null    object 
 7   matricule  1347 non-null   object 
 8   sang       1347 non-null   object 
 9   assurance  49 non-null     float64
dtypes: float64(1), int64(1), object(8)
memory usage: 105.4+ KB


In [110]:
length_counts = (
    df_membres['matricule']
    .astype(str)
    .str.len()
    .value_counts()
    .sort_index()
    .rename_axis('length')
    .reset_index(name='count')
)

print(length_counts)

   length  count
0       2      1
1       3      1
2       4      1
3       5      2
4       6     38
5       7   1013
6       8    287
7      11      1
8      13      2
9      16      1


In [111]:
df_membres.loc[df_membres['nom'] == 'ABDELHAMID']

,id,nom,prenom,telephone,adresse,sexe,naissance,matricule,sang,assurance
1222,1245,ABDELHAMID,ZAKARIA,None,None,homme,None,5228613,A+,NaN
1309,1332,ABDELHAMID,FARAH,None,None,femme,2009-10-29,7492383,A+,NaN


In [112]:
df_inscriptions.head()

,id,debut,fin,reste,nbsseance,membre,abonnement,etat,total,remise,nbrmois,versement,created_at,updated_at,type,remarque,activities,assurance,user,tripode,see
0,9,2025-01-04,2025-02-04,0,8,7,28,0,2000,0.0,1,2000.0,2025-01-04 12:39:49,2026-08-22 21:31:57,None,None,null,None,53,1,None
1,10,2025-01-04,2025-02-04,0,16,8,35,0,3000,0.0,1,3000.0,2025-01-04 16:23:59,2026-08-22 21:31:57,None,None,null,None,53,1,None
2,11,2025-01-04,2025-02-04,6,28,9,37,0,3500,0.0,1,3500.0,2025-01-04 16:25:47,2026-08-22 21:31:57,None,None,null,None,53,1,None
3,12,2025-01-04,2025-02-04,0,12,10,29,0,2500,0.0,1,2500.0,2025-01-04 17:17:12,2026-08-22 21:31:57,None,None,null,None,53,1,None
4,13,2025-01-04,2025-02-04,0,12,11,29,0,2500,0.0,1,2500.0,2025-01-04 17:18:34,2026-08-22 21:31:57,None,None,null,None,53,1,None


In [113]:
df_inscriptions.loc[df_inscriptions['total'] != df_inscriptions['versement']]

,id,debut,fin,reste,nbsseance,membre,abonnement,etat,total,remise,nbrmois,versement,created_at,updated_at,type,remarque,activities,assurance,user,tripode,see
87,95,2025-02-01,2025-03-01,0,16,8,35,0,3000,0.0,1,0.0,2025-02-01 19:59:55,2026-08-22 21:31:57,None,None,null,None,53,1,None
102,109,2025-02-05,2025-03-05,3,12,101,44,0,2000,0.0,1,0.0,2025-02-05 20:32:30,2026-08-22 21:31:57,None,None,null,None,53,1,None
122,130,2025-02-15,2025-03-15,9,16,113,45,0,2500,0.0,1,2000.0,2025-02-15 20:11:00,2026-08-22 21:31:57,None,None,null,None,53,1,None
192,202,2025-03-08,2025-04-08,12,28,158,30,0,4000,0.0,1,3000.0,2025-03-08 15:57:09,2026-08-22 21:31:57,None,None,null,None,56,1,None
212,222,2025-03-14,2025-04-14,11,12,170,29,0,2500,0.0,1,0.0,2025-03-14 21:49:28,2026-08-22 21:31:57,None,None,null,None,56,1,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3127,3160,2026-07-13,2026-08-13,0,16,1193,35,0,3000,0.0,1,2500.0,2026-07-13 20:29:25,2026-08-22 21:31:57,None,None,None,None,56,1,None
3168,3200,2026-07-20,2026-08-20,11,16,333,35,0,3000,0.0,1,1600.0,2026-07-20 14:24:33,2026-08-22 21:31:57,None,None,null,None,56,1,None
3178,3265,2026-07-29,2026-08-29,8,12,1330,29,1,2500,0.0,1,2000.0,2026-07-29 09:37:22,2026-08-05 08:42:04,None,None,null,None,57,1,None
3230,3262,2026-07-28,2026-08-28,6,8,1046,55,1,2500,0.0,1,0.0,2026-07-28 16:46:20,2026-08-04 17:00:23,None,C/NATIONAL/ C/MEDICAL,null,None,57,1,None


In [114]:
df_inscriptions.drop(columns=['type', 'remarque', 'activities', 'assurance', 'user', 'tripode', 'see', 'created_at', 'updated_at'], inplace=True)

In [115]:
df_inscriptions['membre'] = df_inscriptions['membre'].astype(int)

In [116]:
df_combined = pd.merge(
    df_membres,
    df_inscriptions[df_inscriptions['etat'] == '1'],
    left_on='id',
    right_on='membre',
    how='inner'
)

Also pull in members who currently have **no** active (`etat=1`) subscription, as long as they still subscribed recently: take each member's *latest* subscription starting on/after **2025-08-01** — if that latest one is `etat=0`, they were dropped by the merge above, so add them using that latest subscription's data. If their latest recent subscription is `etat=1`, they're already in `df_combined` — skip. A member already present in `df_combined` (e.g. an older `etat=1` row exists even if it's not their most recent) is left as-is rather than overwritten, so nobody is double-imported.

In [117]:
df_inscriptions['debut'] = pd.to_datetime(df_inscriptions['debut'], format='%Y-%m-%d %H:%M:%S')

In [118]:
THRESHOLD_DATE = '2025-08-01'

recent = df_inscriptions[df_inscriptions['debut'] >= THRESHOLD_DATE]
latest_recent = recent.sort_values('debut').groupby('membre', as_index=False).tail(1)
inactive_latest = latest_recent[latest_recent['etat'] == '0']

df_inactive = pd.merge(
    df_membres,
    inactive_latest,
    left_on='id',
    right_on='membre',
    how='inner',
)
candidates = len(df_inactive)
df_inactive = df_inactive[~df_inactive['id_x'].isin(df_combined['id_x'])]
print(f"Adding {len(df_inactive)} inactive member(s) whose latest subscription since {THRESHOLD_DATE} is etat=0 "
      f"({candidates - len(df_inactive)} already covered by df_combined and left untouched).")

df_combined = pd.concat([df_combined, df_inactive], ignore_index=True)

Adding 896 inactive member(s) whose latest subscription since 2025-08-01 is etat=0 (0 already covered by df_combined and left untouched).


In [119]:
df_combined.head()

,id_x,nom,prenom,telephone,adresse,sexe,naissance,matricule,sang,assurance,id_y,debut,fin,reste,nbsseance,membre,abonnement,etat,total,remise,nbrmois,versement
0,8,AHMED,AHMED,0655621426,None,homme,2008-05-02,16175223,A+,NaN,3359,2026-08-15,2026-09-15,12,16,8,35,1,2500,500.0,1,2500.0
1,12,TAIB,BILEL,0560185905,None,homme,2002-10-03,8346324,A+,NaN,3289,2026-08-02,2026-09-02,6,12,12,29,1,2000,500.0,1,2000.0
2,14,BELKADI,RAYAN,None,None,homme,2011-01-28,3539614,A+,NaN,3293,2026-08-03,2026-09-03,7,16,14,35,1,3000,0.0,1,3000.0
3,16,ZEKRI,AYOUB,None,None,homme,2003-12-01,7348978,A+,NaN,3386,2026-08-20,2026-09-20,10,12,16,29,1,2000,500.0,1,2000.0
4,36,SEBTI,YACINE,None,None,homme,2002-05-05,14541998,A+,NaN,3235,2026-07-25,2026-08-25,1,12,36,29,1,2500,0.0,1,2500.0


In [120]:
df_combined['sang'].value_counts()

sang
A+     649
O+     238
B+      99
AB+     34
O-      21
A-      20
B-      17
AB-      2
Name: count, dtype: int64

In [121]:
df_combined['assurance'].unique()

array([nan,  1.])

In [122]:
df_combined['assurance'].fillna(0 , inplace=True)

C:\Users\zakmins\AppData\Local\Temp\ipykernel_15856\2061064533.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_combined['assurance'].fillna(0 , inplace=True)


In [123]:
df_combined['assurance'] = df_combined['assurance'].astype(int)

## Write df_combined into the live Smolympic SQLite database

**Close the Smolympic app before running the next cell** — it holds `smolympic.db` open, and writing to it from here while the app is running can lock the file or leave the app showing stale data until restarted.

Safe to re-run: a row whose `matricule` already exists as an `rfid_uid` in the database is skipped (already migrated by an earlier run) — no duplicates.

Maps the legacy `gym` MySQL fields onto Smolympic's `members` schema:

| df_combined column(s) | Smolympic field |
|---|---|
| `nom`, `prenom` | `name` (title-cased, "Prénom Nom") |
| `sexe` (`homme`/`femme`) | `gender` (`M`/`F`) |
| `naissance` | `dob` |
| `telephone` | `phone` |
| `matricule` | `rfid_uid` — each member's RFID tag code, left-padded with zeros to 10 digits |
| `sang` | `blood_type` |
| `assurance` | `insurance` (+ `insurance_expiry` = sub_start + 365d) |
| `debut`, `fin` | `sub_start`, `sub_end` |
| `nbrmois` | `duration_days` (× 30) |
| `nbsseance`, `reste` | `sessions_total`, `sessions_left` |
| `versement` | recorded as a `payments` row (revenue history) |
| — | `balance` = 0 for every imported member — everyone is treated as fully paid up at import time, no debt tracked (`total`/`versement` are not used to compute it; `remise` is ignored too) |

`id_x`, `id_y`, `membre`, `abonnement`, `etat`, `adresse`, `total` have no matching column in Smolympic and are **not** imported. `sports`/`membership_type` are defaulted to `["GYM"]`/`"subscription"` since the legacy schema has no per-sport data. A member with a missing or duplicate `matricule` falls back to an auto-generated placeholder tag (same scheme Smolympic itself uses), flagged in the summary printout.

In [124]:
import os, shutil, sqlite3, random
from datetime import datetime, timedelta

INSURANCE_DAYS = 365
GENDER_MAP = {'homme': 'M', 'femme': 'F'}
BLOOD_TYPES = {'A+', 'A-', 'B+', 'B-', 'AB+', 'AB-', 'O+', 'O-'}

# Smolympic stores its SQLite DB in Electron's userData folder.
db_path = os.path.expandvars(r'%APPDATA%\SMOLYMPIC\smolympic.db')
assert os.path.exists(db_path), f"Smolympic database not found at {db_path}"

# Always back up the live database before writing to it — this import is a
# one-shot bulk write straight to SQLite, bypassing the app entirely.
backup_path = f"{db_path}.bak-{datetime.now():%Y%m%d%H%M%S}"
shutil.copy2(db_path, backup_path)
print(f"Backed up live database to {backup_path}")


def clean_str(v):
    if v is None or pd.isna(v):
        return None
    v = str(v).strip()
    return v or None


def date_only(v):
    """-> 'YYYY-MM-DD', matching how Smolympic stores sub_start/sub_end."""
    if v is None or pd.isna(v):
        return None
    return v.strftime('%Y-%m-%d') if hasattr(v, 'strftime') else str(v)[:10]


def timestamp(date_str):
    """-> 'YYYY-MM-DDT00:00:00', matching Smolympic's iso() timestamp fields."""
    return f"{date_str}T00:00:00" if date_str else datetime.now().strftime('%Y-%m-%dT%H:%M:%S')


def clean_num(v, default=0):
    if v is None or pd.isna(v):
        return default
    return float(v)


conn = sqlite3.connect(db_path)
conn.execute('PRAGMA foreign_keys = ON')
cur = conn.cursor()

existing_count = cur.execute('SELECT COUNT(*) FROM members').fetchone()[0]
if existing_count:
    print(f"members table already has {existing_count} row(s) — importing on top of existing data.")

# rfid_uid is UNIQUE NOT NULL — matricule is the real tag code. A matricule that
# already exists as an rfid_uid means this member was already migrated by an
# earlier run of this cell — skip them entirely rather than re-inserting a
# duplicate. A row with no matricule (or one reused across two rows in this
# same df_combined) falls back to a generated placeholder, using the same
# scheme Smolympic's own createMember() uses for an unscanned tag.
already_in_db = {r[0] for r in cur.execute('SELECT rfid_uid FROM members')}
existing_rfids = set(already_in_db)
_rfid_seq = 0


def next_placeholder_rfid():
    global _rfid_seq
    while True:
        _rfid_seq += 1
        candidate = str(4200000000 + _rfid_seq * 13)
        if candidate not in existing_rfids:
            return candidate


dup_ids = df_combined.loc[df_combined['id_x'].duplicated(), 'id_x'].tolist()
if dup_ids:
    print(f"Warning: {len(dup_ids)} member id(s) appear more than once in df_combined: {dup_ids}")

inserted, skipped, bad_gender, already_imported = 0, [], 0, []
missing_rfid, dup_rfid = [], []

try:
    for _, row in df_combined.iterrows():
        matricule_rfid = clean_str(row['matricule'])
        # RFID tags are 10 digits — left-pad a shorter matricule with zeros
        # (e.g. "7919718" -> "0007919718") so it matches the tag's real encoding.
        if matricule_rfid:
            matricule_rfid = matricule_rfid.zfill(10)
        if matricule_rfid and matricule_rfid in already_in_db:
            already_imported.append(row['id_x'])
            continue

        nom, prenom = clean_str(row['nom']), clean_str(row['prenom'])
        if not nom and not prenom:
            skipped.append(row['id_x'])
            continue
        name = ' '.join(p.title() for p in (prenom, nom) if p)

        sexe_key = (clean_str(row['sexe']) or '').lower()
        if sexe_key not in GENDER_MAP:
            bad_gender += 1
        gender = GENDER_MAP.get(sexe_key, 'M')

        dob = date_only(row['naissance'])
        phone = clean_str(row['telephone'])

        blood = clean_str(row['sang'])
        if blood:
            blood = blood.upper()
            if blood not in BLOOD_TYPES:
                blood = None

        if matricule_rfid and matricule_rfid not in existing_rfids:
            rfid = matricule_rfid
        else:
            (dup_rfid if matricule_rfid else missing_rfid).append(row['id_x'])
            rfid = next_placeholder_rfid()
        existing_rfids.add(rfid)

        sub_start = date_only(row['debut'])
        sub_end = date_only(row['fin'])
        months = int(clean_num(row['nbrmois'], 1)) or 1
        duration_days = months * 30

        sessions_total = None if pd.isna(row['nbsseance']) else int(row['nbsseance'])
        sessions_left = None if pd.isna(row['reste']) else int(row['reste'])

        # Every migrated member is treated as fully paid up — nobody owes the gym
        # anything at import time, regardless of what total/versement say.
        paid = max(0, round(clean_num(row['versement'])))
        balance = 0

        insured = bool(clean_num(row['assurance']))
        insurance_expiry = None
        if insured:
            anchor = sub_start or datetime.now().strftime('%Y-%m-%d')
            insurance_expiry = timestamp(
                (datetime.strptime(anchor, '%Y-%m-%d') + timedelta(days=INSURANCE_DAYS)).strftime('%Y-%m-%d')
            )

        # No original registration timestamp survives in df_combined (created_at was
        # dropped earlier) — the active subscription's start date is the closest proxy.
        join_date = timestamp(sub_start)
        hue = random.randint(0, 359)

        cur.execute(
            """INSERT INTO members (rfid_uid,name,gender,dob,phone,blood_type,sports,membership_type,
                 sub_start,sub_end,duration_days,sessions_total,sessions_left,
                 insurance,insurance_expiry,balance,hue,join_date)
               VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)""",
            (rfid, name, gender, dob, phone, blood, '["GYM"]', 'subscription',
             sub_start, sub_end, duration_days, sessions_total, sessions_left,
             int(insured), insurance_expiry, balance, hue, join_date),
        )
        member_id = cur.lastrowid

        if paid > 0:
            cur.execute(
                "INSERT INTO payments (member_id,amount,kind,sport,method,date) VALUES (?,?,?,?,?,?)",
                (member_id, paid, 'subscription', 'GYM', 'Cash', timestamp(sub_start)),
            )
        inserted += 1
except Exception:
    conn.rollback()
    conn.close()
    raise

conn.commit()
conn.close()

print(f"Inserted {inserted} member(s).")
if already_imported:
    print(f"Skipped {len(already_imported)} row(s) already migrated in a previous run (old id_x): {already_imported}")
if skipped:
    print(f"Skipped {len(skipped)} row(s) with no name (old id_x): {skipped}")
if bad_gender:
    print(f"{bad_gender} row(s) had an unrecognized 'sexe' value and were defaulted to gender='M'.")
if missing_rfid:
    print(f"{len(missing_rfid)} row(s) had no matricule and got a generated placeholder tag (old id_x): {missing_rfid}")
if dup_rfid:
    print(f"{len(dup_rfid)} row(s) had a matricule already used elsewhere and got a generated placeholder tag instead (old id_x): {dup_rfid}")
print("Note: 'adresse' has no matching field in Smolympic and was not imported.")
print(f"A backup of the pre-import database was saved to: {backup_path}")


Backed up live database to C:\Users\zakmins\AppData\Roaming\SMOLYMPIC\smolympic.db.bak-20260823162753
Inserted 1080 member(s).
Note: 'adresse' has no matching field in Smolympic and was not imported.
A backup of the pre-import database was saved to: C:\Users\zakmins\AppData\Roaming\SMOLYMPIC\smolympic.db.bak-20260823162753


### One-time fix: zero out balances from the earlier `total + versement` run

The import cell above was previously run with `balance = total + versement`, which inserted 1080 members with an inflated (roughly doubled) `balance` — that's why every member currently shows an outstanding balance. The rule is now: nobody owes anything at import time, so `balance` should simply be 0 for every migrated member. The formula above is fixed for future runs, but the 1080 rows already in the live DB still hold the wrong value and need a one-time correction.

The cell below sets `balance = 0` on every member currently in `df_combined` (matched on the same zero-padded `matricule` → `rfid_uid`) — it does **not** touch members outside `df_combined` (e.g. manually created in the app, who may have a real balance). It backs up the database first, same as the import cell. Safe to re-run.

In [125]:
import os, shutil, sqlite3
from datetime import datetime

db_path = os.path.expandvars(r'%APPDATA%\SMOLYMPIC\smolympic.db')
assert os.path.exists(db_path), f"Smolympic database not found at {db_path}"

backup_path = f"{db_path}.bak-{datetime.now():%Y%m%d%H%M%S}"
shutil.copy2(db_path, backup_path)
print(f"Backed up live database to {backup_path}")


def clean_str(v):
    if v is None or pd.isna(v):
        return None
    v = str(v).strip()
    return v or None


conn = sqlite3.connect(db_path)
cur = conn.cursor()

rfids = set()
for _, row in df_combined.iterrows():
    matricule_rfid = clean_str(row['matricule'])
    if matricule_rfid:
        rfids.add(matricule_rfid.zfill(10))

fixed = 0
for rfid in rfids:
    cur.execute('UPDATE members SET balance=0 WHERE rfid_uid=?', (rfid,))
    fixed += cur.rowcount

conn.commit()
conn.close()

print(f"Zeroed out balance on {fixed} member row(s) (out of {len(rfids)} distinct matricule(s) in df_combined).")

Backed up live database to C:\Users\zakmins\AppData\Roaming\SMOLYMPIC\smolympic.db.bak-20260823162753
Zeroed out balance on 1080 member row(s) (out of 1080 distinct matricule(s) in df_combined).


## Backfill revenue history: one payment per inscription, not per member

`df_combined` (and therefore the member-import cell) keeps **at most one** inscription per member — either their current active one, or their single latest expired one since 2025-08-01. Every other inscription that member ever had (each with its own `versement`) was never turned into a `payments` row, so revenue reports in the app understate history.

This cell fixes that for members **already imported** (i.e. present in `df_combined`, therefore in the `members` table with a real `rfid_uid` — not a placeholder tag): for each such member, it walks *every* row in `df_inscriptions` belonging to them (any `etat`, any date) and inserts a `payments` row for each one that isn't already covered by the member-import cell, using that inscription's own `debut` as the date and `versement` as the amount.

Per your call: members whose latest inscription is older than 2025-08-01 are **not** included here — they were never imported as members, so there's no `member_id` to attach their old payments to. Their historical revenue stays unrecorded.

Matching is done by re-deriving `rfid_uid` from `matricule` the same way the import cell did; a member whose matricule collided with another (and so got a generated placeholder tag instead) can't be reliably re-matched here and is skipped, reported at the end (there were none in the last import run).

To make re-running less risky (no natural unique key on `payments`), it skips any inscription for which a `payments` row already exists with the same `member_id`, `amount`, and `date` — a good-enough guard, not a hard guarantee. Backs up the database first.

In [126]:
import os, shutil, sqlite3
from datetime import datetime

db_path = os.path.expandvars(r'%APPDATA%\SMOLYMPIC\smolympic.db')
assert os.path.exists(db_path), f"Smolympic database not found at {db_path}"

backup_path = f"{db_path}.bak-{datetime.now():%Y%m%d%H%M%S}"
shutil.copy2(db_path, backup_path)
print(f"Backed up live database to {backup_path}")


def clean_str(v):
    if v is None or pd.isna(v):
        return None
    v = str(v).strip()
    return v or None


def clean_num(v, default=0):
    if v is None or pd.isna(v):
        return default
    return float(v)


def date_only(v):
    if v is None or pd.isna(v):
        return None
    return v.strftime('%Y-%m-%d') if hasattr(v, 'strftime') else str(v)[:10]


def timestamp(date_str):
    return f"{date_str}T00:00:00" if date_str else None


conn = sqlite3.connect(db_path)
cur = conn.cursor()

rfid_to_member_id = {r[0]: r[1] for r in cur.execute('SELECT rfid_uid, id FROM members')}

# Re-run guard: skip an inscription if a payment already matching it exists.
existing_payments = {
    (r[0], r[1], r[2])
    for r in cur.execute("SELECT member_id, amount, date FROM payments WHERE kind='subscription'")
}

inserted, already_present, no_versement, unmatched_member = 0, 0, 0, []

for _, row in df_combined.iterrows():
    matricule = clean_str(row['matricule'])
    if not matricule:
        continue
    rfid = matricule.zfill(10)
    member_id = rfid_to_member_id.get(rfid)
    if member_id is None:
        unmatched_member.append(row['id_x'])
        continue

    already_covered_inscription_id = row['id_y']
    member_inscriptions = df_inscriptions[df_inscriptions['membre'] == row['id_x']]

    for _, insc in member_inscriptions.iterrows():
        if insc['id'] == already_covered_inscription_id:
            continue  # already inserted by the member-import cell

        amount = max(0, round(clean_num(insc['versement'])))
        if amount <= 0:
            no_versement += 1
            continue

        date_str = timestamp(date_only(insc['debut']))
        key = (member_id, amount, date_str)
        if key in existing_payments:
            already_present += 1
            continue

        cur.execute(
            "INSERT INTO payments (member_id,amount,kind,sport,method,date) VALUES (?,?,?,?,?,?)",
            (member_id, amount, 'subscription', 'GYM', 'Cash', date_str),
        )
        existing_payments.add(key)
        inserted += 1

conn.commit()
conn.close()

print(f"Inserted {inserted} backfilled payment(s) from prior inscriptions.")
if already_present:
    print(f"{already_present} inscription(s) skipped — a matching payment already existed (likely a prior run of this cell).")
if no_versement:
    print(f"{no_versement} inscription(s) had no versement (0 or missing) and were skipped.")
if unmatched_member:
    print(f"{len(unmatched_member)} member(s) could not be matched to a real rfid_uid (placeholder-tagged) and were skipped (old id_x): {unmatched_member}")

Backed up live database to C:\Users\zakmins\AppData\Roaming\SMOLYMPIC\smolympic.db.bak-20260823162753
Inserted 1904 backfilled payment(s) from prior inscriptions.
5 inscription(s) skipped — a matching payment already existed (likely a prior run of this cell).
13 inscription(s) had no versement (0 or missing) and were skipped.
